# TerraLens: Dando una nueva visión al reciclaje.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import regularizers

In [ ]:
# Configuración de parámetros
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_PATH = "~/TerraLens/dataset-residuos"

In [ ]:
# Carga de datos con split automático (80% tren, 20% validación)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

In [ ]:
#Imprimiendo las lases
print(train_ds.class_names)

In [ ]:
# Optimización de rendimiento
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [16]:
def alterar_colores(imagen):
    # Cambia los tonos (ej. un fondo blanco podría verse verdoso o rojizo a veces)
    imagen = tf.image.random_hue(imagen, max_delta=0.15)
    # Cambia la intensidad (hace los colores desteñidos o súper radiactivos)
    imagen = tf.image.random_saturation(imagen, lower=0.5, upper=1.5)
    return imagen

data_augmentation = tf.keras.Sequential([
    # Rotaciones y movimientos (para la forma)
    layers.RandomFlip("horizontal_and_vertical"), 
    layers.RandomRotation(factor=1.0),            
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1), 
    layers.RandomZoom(height_factor=(-0.2, 0.2)), 
    
    # LA NUEVA MAGIA: Distorsión de Color y Luz
    layers.Lambda(alterar_colores, name="Inyeccion_Color_Aleatorio"),
    
    # Distorsión de Brillo y Contraste (para que ignore los reflejos blancos)
    layers.RandomBrightness(factor=0.4), # Subimos el factor a 0.4 para ser agresivos
    layers.RandomContrast(factor=0.4),   
])

In [17]:
base_model = applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)

In [ ]:
#Congelamos el modelo base (no se entrena, ya sabe ver formas)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,              # Capa de aumento de datos
    layers.Lambda(applications.resnet50.preprocess_input), # Normalización específica de ResNet
    base_model,                     # El "cerebro" pre-entrenado
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),            # Para evitar que la IA memorice
    layers.Dense(5, activation='softmax')
])


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
)

## Guardar el modelo

In [ ]:
model.save('clasificador_v3.keras')

## Predicción

## Matriz de Confusion

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import os # Import os module to list directories

# 1. Obtener las etiquetas reales (y_true) y las predicciones (y_pred)
y_true = []
y_pred = []

print("Calculando predicciones... (esto puede tardar un poco)")
for images, labels in val_ds:
    # Guardamos las etiquetas reales
    y_true.extend(np.argmax(labels, axis=1))

    # Hacemos la predicción y guardamos el índice con mayor probabilidad
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))

# 2. Configurar los nombres de las clases (en orden alfabético)
# The original train_ds object was modified by .cache().shuffle().prefetch()
# which removes the .class_names attribute. We retrieve them from the directory.
class_names = sorted(os.listdir(DATASET_PATH))

# 3. Crear la matriz
cm = confusion_matrix(y_true, y_pred)

# 4. Graficar usando Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicción de la IA')
plt.ylabel('Etiqueta Real (La Verdad)')
plt.title('Matriz de Confusión: ¿Dónde se equivoca mi modelo?')
plt.show()

# 5. Imprimir reporte de métricas detallado
print("\nReporte de Clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names))